# WGAN-GP（Wasserstein GAN with Gradient Penalty）

---
## 目的
Wasserstein GAN (WGAN) を構築し，通常のGAN（`gan.ipynb`, `dcgan.ipynb`）の学習が不安定になりやすい問題を，損失関数の設計によって緩和する仕組みを理解する．さらに，WGANの学習をより安定させる**Gradient Penalty**という正則化を組み合わせたWGAN-GPを構築し，その効果を理解する．

## モジュールのインポート
はじめに必要なモジュールをインポートしたのち，GPUを使用した計算が可能かどうかを確認する．

In [ ]:
from time import time
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.autograd as autograd
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## WGAN
`gan.ipynb`のGANは，Discriminatorが実画像・生成画像を「本物である確率」として0〜1の値で出力し，binary cross entropyで学習していました．この定式化では，実画像の分布と生成画像の分布が全く重ならない場合（学習初期にありがちな状況）に，Discriminatorの勾配がほぼ0になってしまい，Generatorの学習が進まなくなる問題（勾配消失）が知られています．

WGAN[1]は，2つの確率分布間の距離の指標として，**Wasserstein距離**（Earth Mover's Distance）を用いることで，分布が重なっていない場合でも滑らかな勾配を与え，学習を安定させます．WGANでは，Discriminatorの最終層のSigmoidを取り除いて実数値をそのまま出力させ（この場合，「識別器」ではなく，本物らしさをスコアリングする**Critic**と呼びます），以下の誤差関数を最適化します．

$$
\max_{D} \; \mathbb{E}_{x\sim p_{data}(x)}\left[D(x)\right] - \mathbb{E}_{z\sim p(z)}\left[D(G(z))\right]
$$
$$
\min_{G} \; -\mathbb{E}_{z\sim p(z)}\left[D(G(z))\right]
$$

Wasserstein距離の理論的な導出には，Criticが1-Lipschitz連続である（＝入力の変化量に対する出力の変化量が，常に一定の比率以下に抑えられている）必要があります．元のWGANでは，Criticの重みパラメータを学習の度に一定の範囲$[-c, c]$に収める**Weight Clipping**によって，これを近似的に実現していました．

[1] M. Arjovsky, S. Chintala, L. Bottou, "Wasserstein GAN," ICML, 2017.

## Weight Clippingの問題点とGradient Penalty
Weight Clippingは実装が簡単な反面，以下のような問題が知られています．

* **表現力の低下**：重みを強制的に小さい範囲に収めるため，Criticが学習できる関数の複雑さが大きく制限され，単純な関数（多くの場合，入力に対してほぼ線形な関数）しか学習できなくなりがちです．
* **勾配消失・爆発**：`clip_value`を小さくしすぎると，層を重ねるごとに勾配がどんどん小さくなり（勾配消失），逆に大きくしすぎると，勾配が層を経るごとに増幅されてしまいます（勾配爆発）．適切な`clip_value`を見つけるための調整が難しいという問題があります．

WGAN-GP[2]は，Weight Clippingの代わりに，**Gradient Penalty**という正則化項を誤差関数に追加することで，1-Lipschitz制約をより直接的に満たすようにする手法です．関数が1-Lipschitz連続であることは，「その関数の勾配のノルムが，常に1以下である」こととほぼ同値です．そこでWGAN-GPは，Criticの入力に対する勾配のノルムが1に近づくように，以下の正則化項を誤差関数に追加します．

$$
\lambda_{gp} \cdot \mathbb{E}_{\hat{x}\sim p_{\hat{x}}}\left[\left(\|\nabla_{\hat{x}} D(\hat{x})\|_{2} - 1\right)^{2}\right]
$$

ここで，$\hat{x}$は，実画像$x$と生成画像$G(z)$を結ぶ直線上を一様分布からサンプリングした係数$\epsilon\sim U[0,1]$で線形補間した点，$\hat{x} = \epsilon x + (1-\epsilon) G(z)$です．実画像・生成画像そのものではなく，その間を補間した点で勾配を評価するのは，最適なCriticの勾配のノルムが，実画像と生成画像を結ぶ直線上でほぼ常に1になる，という理論的な性質に基づいています．

[2] I. Gulrajani, F. Ahmed, M. Arjovsky, V. Dumoulin, A. Courville, "Improved Training of Wasserstein GANs," NeurIPS, 2017.

## ネットワークの構築
Generatorは`dcgan.ipynb`と同じ畳み込み構造を使用します（詳細は`dcgan.ipynb`を参照してください）．

Criticも基本的な構造は同じですが，**Batch Normalizationを使用しない**点が異なります．Gradient Penaltyは，ミニバッチ内の**1サンプルごと**にCriticの入力に対する勾配を計算する必要がありますが，Batch Normalizationはミニバッチ内の他のサンプルの統計量に依存する処理であるため，「1サンプルだけを動かしたときの出力の変化」という勾配penaltyの前提と相性がよくありません．そこで，ミニバッチ内の他のサンプルに依存せず，1サンプルごとに独立して正規化を行う**Layer Normalization**（`nn.GroupNorm(1, C)`で実装できます）を代わりに使用します．

In [ ]:
class Generator(nn.Module):
    def __init__(self, latent_dim=100, out_ch=1):
        super().__init__()
        self.model = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, 256, kernel_size=4, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True),  # 1x1 -> 4x4

            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),  # 4x4 -> 8x8

            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),  # 8x8 -> 16x16

            nn.ConvTranspose2d(64, out_ch, kernel_size=4, stride=2, padding=1, bias=False),
            nn.Sigmoid(),  # 16x16 -> 32x32
        )

    def forward(self, z):
        return self.model(z)


class Critic(nn.Module):
    def __init__(self, in_ch=1):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(in_ch, 64, kernel_size=4, stride=2, padding=1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),  # 32x32 -> 16x16

            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1, bias=False),
            nn.GroupNorm(1, 128), nn.LeakyReLU(0.2, inplace=True),  # 16x16 -> 8x8（Layer Normalization）

            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1, bias=False),
            nn.GroupNorm(1, 256), nn.LeakyReLU(0.2, inplace=True),  # 8x8 -> 4x4（Layer Normalization）

            nn.Conv2d(256, 1, kernel_size=4, stride=1, padding=0, bias=False),  # 4x4 -> 1x1（Sigmoidは適用しない）
        )

    def forward(self, x):
        return self.model(x).view(-1)  # スカラー値（本物らしさのスコア）を返す

## Gradient Penaltyの実装
実画像`real`と生成画像`fake`を，サンプルごとに独立な係数$\epsilon$で線形補間した`interpolated`を作成し，`interpolated`に対するCriticの出力の，`interpolated`自身に対する勾配を，`torch.autograd.grad`で計算します．

このとき，Gradient Penalty自体をCriticの学習の誤差関数に含めて逆伝播するため，勾配を計算するための計算グラフ自体も保持しておく必要があります（`create_graph=True`）．

In [ ]:
def gradient_penalty(D, real, fake, device):
    batch = real.size(0)
    eps = torch.rand(batch, 1, 1, 1, device=device)  # サンプルごとに独立な補間係数
    interpolated = (eps * real + (1 - eps) * fake).requires_grad_(True)

    d_interpolated = D(interpolated)
    gradients = autograd.grad(
        outputs=d_interpolated,
        inputs=interpolated,
        grad_outputs=torch.ones_like(d_interpolated),
        create_graph=True,  # Gradient Penalty自体を誤差逆伝播するため，計算グラフを保持する
        retain_graph=True,
    )[0]

    gradients = gradients.view(batch, -1)
    gradient_norm = gradients.norm(2, dim=1)  # サンプルごとの勾配のL2ノルム
    return ((gradient_norm - 1) ** 2).mean()

## データセットと最適化関数
データセットには，`dcgan.ipynb`と同様に$32\times32$にリサイズしたMNISTを使用します．Gradient Penaltyを用いる場合，Weight Clippingのようにモーメンタムを持つ最適化手法を避ける必要がなくなるため，WGAN-GPの原論文にならい，Adam optimizer（$\beta_1=0, \beta_2=0.9$）を使用します．

In [ ]:
transform = transforms.Compose([transforms.Resize((32, 32)), transforms.ToTensor()])
mnist_data = datasets.MNIST(root='./data', train=True, transform=transform, download=True)
train_loader = DataLoader(mnist_data, batch_size=64, shuffle=True)

latent_dim = 100
G = Generator(latent_dim=latent_dim, out_ch=1).to(device)
D = Critic(in_ch=1).to(device)

opt_g = torch.optim.Adam(G.parameters(), lr=1e-4, betas=(0.0, 0.9))
opt_d = torch.optim.Adam(D.parameters(), lr=1e-4, betas=(0.0, 0.9))

## WGAN-GPの学習
Criticは`n_critic`回更新するごとに，Generatorを1回更新します．Criticの誤差関数は，Wasserstein距離の負値に，`lambda_gp`で重み付けしたGradient Penaltyを加えたものです．Weight Clippingは行いません．

In [ ]:
epoch_num = 20
n_critic = 5
lambda_gp = 10

G.train()
D.train()
start = time()
for epoch in range(1, epoch_num + 1):
    critic_count = 0
    for idx, (real_x, _) in enumerate(train_loader):
        real_x = real_x.to(device)
        batch = real_x.size(0)

        # Criticの更新（毎iteration）
        z = torch.randn(batch, latent_dim, 1, 1).to(device)
        fake_x = G(z)

        gp = gradient_penalty(D, real_x, fake_x.detach(), device)
        d_loss = -(D(real_x).mean() - D(fake_x.detach()).mean()) + lambda_gp * gp

        opt_d.zero_grad()
        d_loss.backward()
        opt_d.step()

        # Generatorの更新（Criticをn_critic回更新するごとに1回）
        critic_count += 1
        if critic_count % n_critic == 0:
            z = torch.randn(batch, latent_dim, 1, 1).to(device)
            fake_x = G(z)
            g_loss = -D(fake_x).mean()

            opt_g.zero_grad()
            g_loss.backward()
            opt_g.step()

    wasserstein_estimate = (D(real_x).mean() - D(fake_x.detach()).mean()).item()
    print(f'epoch: {epoch}, Wasserstein距離の推定値: {wasserstein_estimate:.4f}, GP: {gp.item():.4f}, '
          f'G loss: {g_loss.item():.4f}, elapsed_time: {time() - start:.4f}')

## 学習したGeneratorによる画像生成
正規分布に従う乱数`z`を生成し，Generatorへ入力することで画像を生成します．

In [ ]:
num_generate = 100
z = torch.randn(num_generate, latent_dim, 1, 1).to(device)

G.eval()
with torch.no_grad():
    test_img = G(z)

test_img = test_img.view(num_generate, 32, 32).cpu().numpy()

fig = plt.figure(figsize=(10, 10))
for i, im in enumerate(test_img):
    ax = fig.add_subplot(10, 10, i + 1, xticks=[], yticks=[])
    ax.imshow(im, cmap='gray')
plt.show()

## 課題

1. `lambda_gp`の値を大きく・小さく変更して学習し，生成画像の質や学習の安定性がどのように変化するか確認してください．
2. Criticの正規化層を`nn.GroupNorm(1, C)`（Layer Normalization）から`nn.BatchNorm2d`に戻して学習し，Gradient Penaltyとの相性がどのように悪くなるか（学習がうまく進まなくなるか）確認してください．
3. `n_critic`を`1`に変更して学習し，`dcgan.ipynb`のDCGAN（`n_critic=1`）と比べて学習がどの程度安定しているか比較してください．